## Setup Environment

In [ ]:
%conda create -n eupmu-style python=3.10
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118
%pip install xformers
%pip install -r requirements.txt

## Create Configs

### Step 1: Choose Base Model

Examples of `pretrained_sd_model`:
- SD v1.4: `CompVis/stable-diffusion-v1-4`
- SD v1.5: `runwayml/stable-diffusion-v1-5`
- SD v2.1: `stabilityai/stable-diffusion-2-1-base`
- WD1.5 beta3: `Birchlabs/wd-1-5-beta3-unofficial`
- SDXL: `stabilityai/stable-diffusion-xl-base-1.0`

If base model is v2.x, set `is_v2_model` to `true`.

In [1]:
pretrained_sd_model = "CompVis/stable-diffusion-v1-4"  #@param {type: "string"}
is_v2_model = "false" #@param ["true", "false"]
is_v_prediction_model = "false" #@param ["true", "false"]

### Step 2: Choose Concept

- `target_concept`: Targeted concept for erasing
- `surrogate_concept`: Surrogate concept for defining model generation after erasure, empty string by default

In [2]:
target_concept = 'van gogh'  #@param {type: "string"}
surrogate_concept = ''  #@param {type: "string"}

### Step 3: EUPMU Settings

- `mode`: `erase_with_la` is the only option here for EUPMU
- `dim`: default to 1, other values for ablation
- `sampling_batch_size`: indicates how many latent anchors are sampled for each iteration, default to 4, can be reduced if there's not enough VRAM
- `la_strength`: indicates the latent anchoring loss strength that balancing the erasure and preservation, default to 1000 for SD v1.4 models, should be further tuned for other base models for better performance


In [3]:
mode = 'erase_with_la' #@param ["erase_with_la", "erase"]
dim = 1 #@param {type: "number"}
sampling_batch_size = 4 #@param {type: "number"}
la_strength = 1024 #@param {type: "number"}
erasing_scale = 2.0 #@param {type: "number"}

### Step 4: Training Settings

There are two parts for training settings, SD settings and optimization settings.
Notice that `resolution` is set to 512 for SD v1.x, 768 for SD v2.x, and 1024 for SDXL.

In [8]:
resolution = 512 #@param [512, 640, 768, 896, 960, 1024]
max_denoising_steps = 30  #@param {type: "number"}
dynamic_resolution = "true" #@param ["true", "false"]
clip_skip = 1 #@param [1, 2]

batch_size = 1  #@param {type: "number"}
iterations = 1000  #@param {type: "number"}
lr = 2e-4  #@param {type: "number"}
optimizer = "AdamW8bit" #@param {type: "string"}
lr_scheduler = "constant"#"constant" #@param {type: "string"}
lr_warmup_steps = 0 #@param {type: "number"}
lr_scheduler_num_cycles = 0 #@param {type: "number"}
save_per_steps = 1000#500  #@param {type: "number"}
precision = "float32" #@param ["float32", "float16", "bfloat16"]
verbose = "false" #@param ["true", "false"]

### Step 5: (Optional) Tracking Training Details with WandB

You can setup your wandb token to track the training details, including training statistics (e.g. losses, learning rates) and visualizations.
Your wandb token can be retrieved from https://wandb.ai/authorize .

In [9]:
wandb_token = "bd9b593d31ad39d59b45865fac8168e32f691565" #@param {type: "string"}

prompts_to_visualize = []#["a painting in the style of van gogh", "a painting in the style of rembrandt", "a painting in the style of picasso"]  #@param {type: "string"}
generate_num = 1  #@param {type: "number"}

# track target & surrogate by default
prompts_to_visualize = [target_concept, surrogate_concept] + prompts_to_visualize

# login with your wandb token
if wandb_token != "": 
    !wandb login {wandb_token}


wandb: Appending key for api.wandb.ai to your netrc file: /home/toby/.netrc


### Step 6: Generate Config Files

Run the following code block and the config files are automatically generated.

In [10]:
# you can custom these strings to distiguish your different exps
exp_name = target_concept.replace(" ", "_")
if surrogate_concept:
  exp_name += f"_to_{surrogate_concept.replace(' ', '_')}"
save_name = f"{exp_name}"
run_name = f"{exp_name}"

config_file_path = f"configs/{save_name}/config.yaml"
prompts_file_path = f"configs/{save_name}/prompt.yaml"


config_file_content = f"""
prompts_file: "{prompts_file_path}"

pretrained_model:
  name_or_path: "{pretrained_sd_model}"
  v2: {is_v2_model}
  v_pred: {is_v_prediction_model}
  clip_skip: {clip_skip}

network:
  rank: {dim}
  alpha: 1.0

train:
  precision: {precision}
  noise_scheduler: "ddim"
  iterations: {iterations}
  batch_size: {batch_size}
  lr: {lr}
  unet_lr: {lr}
  text_encoder_lr: {0.5 * lr}
  optimizer_type: "{optimizer}"
  lr_scheduler: "{lr_scheduler}"
  lr_warmup_steps: {lr_warmup_steps}
  lr_scheduler_num_cycles: {lr_scheduler_num_cycles}
  max_denoising_steps: {max_denoising_steps}

save:
  name: "{save_name}"
  path: "output/{save_name}"
  per_steps: {save_per_steps}
  precision: {precision}

logging:
  use_wandb: {"true" if wandb_token != "" else "false"}
  interval: 0
  seed: 0
  generate_num: {generate_num}
  run_name: "{run_name}"
  verbose: {verbose}
  prompts: {prompts_to_visualize}

other:
  use_xformers: true
"""

import os
if not os.path.exists(f"./configs/{save_name}"):
  os.makedirs(f"./configs/{save_name}")

with open(config_file_path, "w") as f:
  f.write(config_file_content)

prompts_file_content = f"""
- target: "{target_concept}"
  positive: "{target_concept}"
  unconditional: ""
  neutral: "{surrogate_concept}"
  action: "{mode}"
  guidance_scale: "{erasing_scale}"
  resolution: {resolution}
  batch_size: {batch_size}
  dynamic_resolution: {dynamic_resolution}
  la_strength: {la_strength}
  sampling_batch_size: {sampling_batch_size}
"""

with open(prompts_file_path, "w") as f:
  f.write(prompts_file_content)


## Start Training

In [11]:
!python train_eupmu.py --config_file {config_file_path}


wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: anser (anser-university-of-illinois-urbana-champaign). Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.18.6
wandb: Run data is saved locally in /media/toby/SSD2/linux_workspaces/2025summer_machine_backup/MU/SD_style+instance_unlearn/EUPMU_style+instance_unlearn/wandb/run-20250901_220427-2f9k6v5x
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run van_gogh
wandb: ⭐️ View project at https://wandb.ai/anser-university-of-illinois-urbana-champaign/SPM
wandb: 🚀 View run at https://wandb.ai/anser-university-of-illinois-urbana-champaign/SPM/runs/2f9k6v5x
Loading checkpoint from CompVis/stable-diffusion-v1-4
/home/toby/environment/miniconda3/envs/spm/lib/python3.10/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Do

For SPM:

In [ ]:
!python ./train_spm.py --config_file {config_file_path}

## Mass Training

In [12]:
class Config:
    def __init__(self, target_concept, surrogate_concept='', is_v2_model=False, is_v_prediction_model=False):
        self.target_concept = target_concept
        self.surrogate_concept = surrogate_concept
        self.pretrained_sd_model = "CompVis/stable-diffusion-v1-4"
        self.is_v2_model = is_v2_model
        self.is_v_prediction_model = is_v_prediction_model
        
        # SPM Settings
        self.mode = 'erase_with_la'
        self.dim = 1
        self.sampling_batch_size = 4
        self.la_strength = 1024
        self.erasing_scale = 2.0
        
        # Training Settings
        self.resolution = 512
        self.max_denoising_steps = 30
        self.dynamic_resolution = True
        self.clip_skip = 1
        self.batch_size = 1
        self.iterations = 1000
        self.lr = 2e-4
        self.optimizer = "AdamW8bit"
        self.lr_scheduler = "constant"
        self.lr_warmup_steps = 0
        self.lr_scheduler_num_cycles = 0
        self.save_per_steps = 1500
        self.precision = "float32"
        self.verbose = False
        
        # WandB Tracking
        self.wandb_token = ""
        self.prompts_to_visualize = [target_concept, surrogate_concept]
        self.generate_num = 1
        
        # Auto-generated config file paths
        self.exp_name = self._generate_exp_name()
        self.config_file_path = f"configs/{self.exp_name}/config.yaml"
        self.prompts_file_path = f"configs/{self.exp_name}/prompt.yaml"
        
    def _generate_exp_name(self):
        exp_name = self.target_concept.replace(" ", "_")
        if self.surrogate_concept:
            exp_name += f"_to_{self.surrogate_concept.replace(' ', '_')}"
        return exp_name
    
    def save_configs(self):
        os.makedirs(f"./configs/{self.exp_name}", exist_ok=True)
        
        # Generate config.yaml
        config_content = f"""
prompts_file: "{self.prompts_file_path}"

pretrained_model:
  name_or_path: "{self.pretrained_sd_model}"
  v2: {self.is_v2_model}
  v_pred: {self.is_v_prediction_model}
  clip_skip: {self.clip_skip}

network:
  rank: {self.dim}
  alpha: 1.0

train:
  precision: {self.precision}
  noise_scheduler: "ddim"
  iterations: {self.iterations}
  batch_size: {self.batch_size}
  lr: {self.lr}
  unet_lr: {self.lr}
  text_encoder_lr: {0.5 * self.lr}
  optimizer_type: "{self.optimizer}"
  lr_scheduler: "{self.lr_scheduler}"
  lr_warmup_steps: {self.lr_warmup_steps}
  lr_scheduler_num_cycles: {self.lr_scheduler_num_cycles}
  max_denoising_steps: {self.max_denoising_steps}

save:
  name: "{self.exp_name}"
  path: "output/{self.exp_name}"
  per_steps: {self.save_per_steps}
  precision: {self.precision}

logging:
  use_wandb: {"true" if self.wandb_token else "false"}
  interval: 0
  seed: 0
  generate_num: {self.generate_num}
  run_name: "{self.exp_name}"
  verbose: {self.verbose}
  prompts: {self.prompts_to_visualize}

other:
  use_xformers: true
"""
        with open(self.config_file_path, "w") as f:
            f.write(config_content)

        # Generate prompt.yaml
        prompts_content = f"""
- target: "{self.target_concept}"
  positive: "{self.target_concept}"
  unconditional: ""
  neutral: "{self.surrogate_concept}"
  action: "{self.mode}"
  guidance_scale: "{self.erasing_scale}"
  resolution: {self.resolution}
  batch_size: {self.batch_size}
  dynamic_resolution: {"true" if self.dynamic_resolution else "false"}
  la_strength: {self.la_strength}
  sampling_batch_size: {self.sampling_batch_size}
"""
        with open(self.prompts_file_path, "w") as f:
            f.write(prompts_content)


# Example Usage:
# Just update the `target_concept` and optionally `surrogate_concept` to generate new configs
#config = Config(target_concept="van gogh")
#config.save_configs()


In [13]:
import subprocess

# List of concepts to forget
forget_concepts_list = ["rembrandt", "picasso", "snoopy", "mickey", "spongebob"]

# Iterate through each concept, generate configs, and execute the training script
for concept in forget_concepts_list:
    # Create a configuration for the current concept
    config = Config(target_concept=concept)
    config.save_configs()  # Save the configuration files
    
    # Command to run the training script
    command = f"python train_eupmu.py --config_file {config.config_file_path}"
    
    # Display command for debugging/logging
    print(f"Running command for concept: {concept}")
    print(command)    

    
    # Execute the command
    try:
        subprocess.run(command, check=True, shell=True)
    except subprocess.CalledProcessError as e:
        print(f"Error running command for concept {concept}: {e}")


Running command for concept: rembrandt
python train_eupmu.py --config_file configs/rembrandt/config.yaml
Loading checkpoint from CompVis/stable-diffusion-v1-4


/home/toby/environment/miniconda3/envs/spm/lib/python3.10/site-packages/huggingface_hub/file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Keyword arguments {'upcast_attention': False} are not expected by StableDiffusionPipeline and will be ignored.
`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["id2label"]` will be overriden.
`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["bos_token_id"]` will be overriden.
`text_config_dict` is provided which will be used to initialize `CLIPTextConfig`. The value `text_config["eos_token_id"]` will be overriden.


lora_unet_down_blocks_0_attentions_0_proj_in
lora_unet_down_blocks_0_attentions_0_transformer_blocks_0_attn1_to_q
lora_unet_down_blocks_0_attentions_0_transformer_blocks_0_attn1_to_k
lora_unet_down_blocks_0_attentions_0_transformer_blocks_0_attn1_to_v
lora_unet_down_blocks_0_attentions_0_transformer_blocks_0_attn1_to_out_0
lora_unet_down_blocks_0_attentions_0_transformer_blocks_0_attn2_to_q
lora_unet_down_blocks_0_attentions_0_transformer_blocks_0_attn2_to_k
lora_unet_down_blocks_0_attentions_0_transformer_blocks_0_attn2_to_v
lora_unet_down_blocks_0_attentions_0_transformer_blocks_0_attn2_to_out_0
lora_unet_down_blocks_0_attentions_0_transformer_blocks_0_ff_net_0_proj
lora_unet_down_blocks_0_attentions_0_transformer_blocks_0_ff_net_2
lora_unet_down_blocks_0_attentions_0_proj_out
lora_unet_down_blocks_0_attentions_1_proj_in
lora_unet_down_blocks_0_attentions_1_transformer_blocks_0_attn1_to_q
lora_unet_down_blocks_0_attentions_1_transformer_blocks_0_attn1_to_k
lora_unet_down_blocks_0_att

100%|██████████| 17/17 [00:01<00:00,  8.94it/s]


losses before weight update 0.0, 0.0026241180021315813, weighted loss: 0.0026241180021315813, weights: [0.]
gradient:  tensor([-0.0032]) tensor(0.) tensor(0.0002)


...

losses before weight update 0.003737187013030052, 0.0016685998998582363, weighted loss: 0.002192786429077387, weights: [0.3394111]
gradient:  tensor([-0.0024]) tensor(0.0037) tensor(0.0031)


100%|██████████| 19/19 [00:01<00:00, 13.04it/s]


losses before weight update 0.0012002851581200957, 0.0031979854684323072, weighted loss: 0.0027089668437838554, weights: [0.3241364]
gradient:  tensor([-0.0028]) tensor(0.0012) tensor(0.0010)
Saving...


Loss*1k: 4.3983: 100%|██████████| 1000/1000 [38:41<00:00,  2.32s/it]


Done.
